# param-group-dict-list — ex2: three-group split — no-decay biases + decay-weights + head LR

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `param-group-dict-list`. Running the final beacon cell reports progress against the `Config: param-group dict list` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: param-group dict list` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`param-group-dict-list`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "param-group-dict-list"
DD_SUBTOPIC = "Config: param-group dict list"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Param groups — three-way split for no-decay biases

Ex1 built two groups by MODULE (encoder vs head) with different LRs. The standard production-quality split is BY PARAMETER ROLE WITHIN A MODULE: weights get weight_decay, biases and normalization params don't. Combined with ex1's encoder/head LR split that becomes a three-group setup:

```python
encoder_decay, encoder_no_decay = split_params(encoder)
head_params = list(head.parameters())

groups = [
    {'params': encoder_decay,    'lr': enc_lr, 'weight_decay': wd},
    {'params': encoder_no_decay, 'lr': enc_lr, 'weight_decay': 0.0},
    {'params': head_params,      'lr': head_lr, 'weight_decay': wd},
]
```

**The split rule.** A param is 'no-decay' if `p.dim() <= 1` (biases are rank-1; weight matrices are rank-2+). LayerNorm weight is rank-1 too, so the same rule catches it. This is the convention used by transformers / vit / nanoGPT.

**Why three groups, not two.** Two groups (decay vs no-decay) loses the encoder/head LR distinction; two groups (encoder vs head) loses the decay distinction. The full transfer-learning fine-tune wants BOTH knobs.

**Iteration count check.** Sum of `len(g['params'])` across all groups MUST equal the total parameter count of the model — otherwise some params silently aren't being trained.

### Exercise 2 — three-group split — no-decay biases + decay-weights + head LR

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the `[{'params': ..., 'lr': ..., 'weight_decay': ...}, ...]` construction to split a model into THREE param groups (encoder-decay, encoder-no-decay, head) using `p.dim() > 1` as the decay-eligibility rule.
> Keywords: param-groups, no-decay-biases, weight-decay, fine-tuning
> ```

**KCs targeted:** `rank-based-param-split`, `param-group-multi-hparam-override`

Implement `ex2_make_three_groups(encoder, head, encoder_lr, head_lr, weight_decay)`.

Build a 3-element list of param-group dicts:
1. **encoder-decay**: encoder params with   `p.dim() > 1` (weight matrices), `lr=encoder_lr`,   `weight_decay=weight_decay`.
2. **encoder-no-decay**: encoder params with   `p.dim() <= 1` (biases, LayerNorm scale),   `lr=encoder_lr`, `weight_decay=0.0`.
3. **head**: ALL head params (no further split),   `lr=head_lr`, `weight_decay=weight_decay`.

Order matters: encoder-decay first, then encoder-no-decay, then head.

Constraints:
- Sum of `len(g['params'])` across the 3 groups must   equal total params in `encoder` + `head` (no param   silently dropped, none duplicated).
- Empty group lists are allowed (e.g. an encoder with   no biases produces an empty group 2).

In [ ]:
def ex2_make_three_groups(encoder, head, encoder_lr: float,
                         head_lr: float, weight_decay: float):
    """Three-group param-list split: decay/no-decay encoder + head."""
    raise NotImplementedError()


def _test_ex2():
    import torch.nn as nn

    # Encoder: 2 Linears (4 params: 2 weights rank-2, 2 biases rank-1).
    encoder = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 16))
    head = nn.Linear(16, 3)   # 1 weight (rank 2) + 1 bias (rank 1)

    groups = ex2_make_three_groups(
        encoder, head,
        encoder_lr=1e-4, head_lr=1e-2, weight_decay=0.01,
    )
    assert isinstance(groups, list), f'must return a list, got {type(groups).__name__}'
    assert len(groups) == 3, f'must return 3 groups, got {len(groups)}'

    # === Each group is a dict with the right keys ===
    for i, g in enumerate(groups):
        assert isinstance(g, dict)
        for key in ['params', 'lr', 'weight_decay']:
            assert key in g, f'group {i} missing {key!r}'

    # === Hparam assignment ===
    # group 0 = encoder-decay
    assert groups[0]['lr'] == 1e-4
    assert groups[0]['weight_decay'] == 0.01
    # group 1 = encoder-no-decay
    assert groups[1]['lr'] == 1e-4
    assert groups[1]['weight_decay'] == 0.0
    # group 2 = head
    assert groups[2]['lr'] == 1e-2
    assert groups[2]['weight_decay'] == 0.01

    # === Rank-based split: only rank>1 params in decay groups ===
    for p in groups[0]['params']:
        assert p.dim() > 1, f'decay group should only have rank>1; got dim={p.dim()}'
    for p in groups[1]['params']:
        assert p.dim() <= 1, f'no-decay group should only have rank<=1; got dim={p.dim()}'

    # === Counts match the model structure ===
    # Encoder: 2 weight matrices (rank-2), 2 biases (rank-1).
    assert len(groups[0]['params']) == 2, f'encoder-decay count: {len(groups[0]["params"])}'
    assert len(groups[1]['params']) == 2, f'encoder-no-decay count: {len(groups[1]["params"])}'
    # Head: 1 weight + 1 bias = 2 params total.
    assert len(groups[2]['params']) == 2, f'head count: {len(groups[2]["params"])}'

    # === Sum of params equals total model params ===
    total_split = sum(len(g['params']) for g in groups)
    total_model = len(list(encoder.parameters())) + len(list(head.parameters()))
    assert total_split == total_model, (
        f'param count mismatch: split has {total_split}, model has {total_model}'
    )

    # === Plug into a real optimizer; per-group hparams round-trip ===
    opt = t.optim.AdamW(groups)
    assert len(opt.param_groups) == 3
    assert opt.param_groups[0]['weight_decay'] == 0.01
    assert opt.param_groups[1]['weight_decay'] == 0.0
    assert opt.param_groups[2]['weight_decay'] == 0.01
    assert opt.param_groups[0]['lr'] == 1e-4
    assert opt.param_groups[2]['lr'] == 1e-2

    # === No duplicates: every param identity appears in exactly one group ===
    all_param_ids = []
    for g in groups:
        for p in g['params']:
            all_param_ids.append(id(p))
    assert len(all_param_ids) == len(set(all_param_ids)), 'param appears in multiple groups'

    # === Encoder without biases (LayerNorm-only, or biases=False) produces empty group 2 ===
    encoder_no_bias = nn.Sequential(nn.Linear(8, 16, bias=False), nn.Linear(16, 16, bias=False))
    head2 = nn.Linear(16, 3)
    groups2 = ex2_make_three_groups(encoder_no_bias, head2, 1e-4, 1e-2, 0.01)
    assert len(groups2[1]['params']) == 0, (
        f'bias-less encoder should produce empty no-decay group, got {len(groups2[1]["params"])}'
    )
    # Decay group still has the 2 weight matrices.
    assert len(groups2[0]['params']) == 2
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_make_three_groups(encoder, head, encoder_lr,
                         head_lr, weight_decay):
    enc_params = list(encoder.parameters())
    head_params = list(head.parameters())
    enc_decay = [p for p in enc_params if p.dim() > 1]
    enc_no_decay = [p for p in enc_params if p.dim() <= 1]
    return [
        {'params': enc_decay,    'lr': encoder_lr, 'weight_decay': weight_decay},
        {'params': enc_no_decay, 'lr': encoder_lr, 'weight_decay': 0.0},
        {'params': head_params,  'lr': head_lr,    'weight_decay': weight_decay},
    ]
```

**Why `p.dim() > 1` as the decay rule.** Biases are rank-1 (shape `(out_features,)`), LayerNorm `weight` is rank-1 (shape `(features,)`), and BatchNorm `weight` / `bias` are rank-1. Weight MATRICES are rank-2 or higher (Linear weight is `(out, in)`; Conv2d weight is `(out, in, kh, kw)`). The `> 1` rule cleanly separates scale/shift params from learned-projection params.

**Why the head isn't split.** Conventionally, the head is small enough that splitting weight-decay inside it doesn't measurably help — and a fresh head is initialized with random values that wants the regularize. If you DO want a four-group split (head-decay + head-no-decay), the same rule applies — just two more comprehensions.

**Empty groups are FINE.** Empty `'params': []` is valid input to `torch.optim` — the group is just a no-op every step. Letting an empty group survive (rather than special-casing it out) keeps the code structure regular and lets logging code count three groups every time.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()